# Part 1 · Prepare scRNA-seq input
Choose synthetic, public PBMC3k or your own annotated counts. This software prototype uses declared rates, not inferred clinical dynamics. Read `docs/02_your_own_data.md` before selecting your own data. Run cells in order.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

# In Colab, the public GitHub version must already contain this release.
INSTALL_PACKAGE = True
REPO_URL = "https://github.com/hylixinsights/VirtualTissue-scRNAseq.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((d for d in candidates if (d / "pyproject.toml").exists() and (d / "src/vtsc").exists()), None)
if ROOT is None:
    ROOT = Path("/content/VirtualTissue-scRNAseq") if IN_COLAB else Path.home() / "VirtualTissue-scRNAseq"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
if INSTALL_PACKAGE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[notebooks]"], check=True)
sys.path.insert(0, str(ROOT / "src"))
print("Code:", ROOT)
# Leave WORK below local, or explicitly mount Drive and choose a Drive path.
WORK = Path(os.environ.get("VTSC_NOTEBOOK_WORK", str(ROOT / "outputs/notebook_demo")))
WORK.mkdir(parents=True, exist_ok=True)
print("Files:", WORK)


## Select input and model
Use `synthetic` first. `pbmc3k` downloads the verified CC BY 4.0 example. For `own`, specify raw-count H5AD/Matrix Market, annotations and your model. The output filename must be new. No folder is uploaded automatically.

In [ ]:
DATA_SOURCE = "synthetic"  # synthetic | pbmc3k | own
OWN_INPUT = Path("/path/to/my_cells.h5ad")
OWN_METADATA = None  # e.g. Path('/path/to/metadata.tsv') for Matrix Market
COUNT_LAYER = None  # 'counts' when H5AD.X contains normalized expression
STATE_COLUMN = 'state'
SAMPLE_COLUMN = 'sample_id'
SAMPLE_ID = None  # required if more than one sample exists
MODEL_PATH = None  # Path('/path/to/reviewed_model.yaml'); None uses demo assumptions
MAX_CELLS = None  # optional explicit uniform subsample, e.g. 1000
SEED = 21
BUNDLE = WORK / 'input.zip'


In [ ]:
from vtsc.inputs import synthetic_example, download_pbmc, prepare, read_bundle
if BUNDLE.exists():
    raise FileExistsError('Bundle already exists. Reuse it in Part 2 or select a new WORK/BUNDLE.')
if DATA_SOURCE == 'synthetic':
    source = WORK / 'synthetic.h5ad'
    if not source.exists(): synthetic_example(source, cells=300, seed=SEED)
    info = prepare(source, BUNDLE, source_license='MIT; generated synthetic fixture, no patient data', max_cells=MAX_CELLS, seed=SEED)
elif DATA_SOURCE == 'pbmc3k':
    source = download_pbmc(WORK / 'pbmc3k')['matrix_directory']
    info = prepare(source, BUNDLE, recipe='pbmc-cd8-demo', source_license='CC-BY-4.0; 10x Genomics PBMC3k, 2016', max_cells=MAX_CELLS, seed=SEED)
elif DATA_SOURCE == 'own':
    info = prepare(OWN_INPUT, BUNDLE, metadata=OWN_METADATA, layer=COUNT_LAYER, model_path=MODEL_PATH,
                   state_column=STATE_COLUMN, sample_column=SAMPLE_COLUMN, sample_id=SAMPLE_ID, max_cells=MAX_CELLS, seed=SEED)
else:
    raise ValueError('Unknown DATA_SOURCE')
info


In [ ]:
cells, model, provenance = read_bundle(BUNDLE)
print('Verified bundle:', BUNDLE)
print('Initial states:', {s: sum(c['state'] == s for c in cells) for s in model['states']})
print('Rates:', model['rate_provenance'])
print('Next: open Part2-3_Run_Simulation.ipynb and use this exact BUNDLE path.')


Keep the input ZIP. It contains states/IDs and provenance but no expression matrix. In Colab, copy it to your own Drive or download it before the session ends. A healthy PBMC example does not validate melanoma or treatment predictions.